# LeetCode 658 — Find K Closest Elements bằng Python

Link đề: https://leetcode.com/problems/find-k-closest-elements/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting, Sliding Window, Heap

Skeleton LeetCode:

```python
class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        ...
```

Ghi chú: bài cho mảng **đã sắp xếp**. Chính tính chất này làm đáp án luôn là **một đoạn liên tiếp (cửa sổ) dài `k`**, nên bài chuyển thành "chọn điểm bắt đầu của cửa sổ", và điểm bắt đầu tốt nhất tìm được bằng **binary search** (tìm kiếm nhị phân). Notebook đi từ cách sắp xếp theo khoảng cách, qua heap, hai con trỏ thu hẹp, đến hai cách dùng binary search.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng đã sắp xếp `arr`, chọn `k` phần tử **gần `x` nhất** (gần theo `|a - x|`, hòa thì phần tử **nhỏ hơn** gần hơn), rồi trả về chúng theo thứ tự **tăng dần**.
2. Nắm các công cụ Python: `sorted(..., key=...)` với khóa là tuple, `abs`, `heapq` (`heappush`, `heapreplace`, `nsmallest`), cắt lát `arr[l:r]`, `bisect.bisect_left`, và khuôn mẫu binary search `while left < right`.
3. Đi từ **sắp xếp theo khoảng cách** → **heap kích thước `k`** → **hai con trỏ thu hẹp hai đầu** → **binary search điểm bắt đầu cửa sổ** → **bisect rồi nở ra hai phía**.
4. Hiểu mấu chốt: đáp án là cửa sổ liên tục; so sánh cửa sổ bắt đầu ở `l` với cửa sổ bắt đầu ở `l + 1` chỉ cần so `x - arr[l]` với `arr[l + k] - x`, và kết quả so sánh này **đơn điệu theo `l`** nên binary search được.
5. Tránh các lỗi kinh điển: dùng `<` thay vì `<=` (sai khi hòa), đặt `right = len(arr) - 1` làm `mid + k` vượt mảng, lấy cửa sổ đối xứng quanh vị trí chèn của `x`, và quên sắp xếp lại kết quả khi chọn theo khoảng cách.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` **đã sắp xếp tăng dần** và hai số nguyên `k`, `x`. Trả về `k` số trong `arr` **gần `x` nhất**. Kết quả cũng phải **sắp xếp tăng dần**.

Số nguyên `a` được coi là **gần `x` hơn** `b` nếu:

- `|a - x| < |b - x|`, hoặc
- `|a - x| == |b - x|` và `a < b`.

**Ví dụ 1**

- Input: `arr = [1,2,3,4,5]`, `k = 4`, `x = 3`
- Output: `[1,2,3,4]`

**Ví dụ 2**

- Input: `arr = [1,1,2,3,4,5]`, `k = 4`, `x = -1`
- Output: `[1,1,2,3]`

**Ràng buộc:**

- `1 <= k <= arr.length`
- `1 <= arr.length <= 10^4`
- `arr` sắp xếp **tăng dần** (có thể có phần tử trùng nhau)
- `-10^4 <= arr[i], x <= 10^4`

Chú ý ví dụ 1: `x = 3` và các khoảng cách là `2,1,0,1,2`. Phần tử `1` (cách 2) và `5` (cách 2) **hòa** nhau, luật hòa chọn số nhỏ hơn nên giữ `1` và bỏ `5`.

In [1]:
from typing import List
from bisect import bisect_left
from itertools import combinations
import heapq
import random
import time

examples = [
    (([1, 2, 3, 4, 5], 4, 3), [1, 2, 3, 4]),
    (([1, 1, 2, 3, 4, 5], 4, -1), [1, 1, 2, 3]),
]
for (arr, k, x), expected in examples:
    print("arr =", arr, "| k =", k, "| x =", x, "| khoảng cách:", [abs(v - x) for v in arr], "| kỳ vọng:", expected)

arr = [1, 2, 3, 4, 5] | k = 4 | x = 3 | khoảng cách: [2, 1, 0, 1, 2] | kỳ vọng: [1, 2, 3, 4]
arr = [1, 1, 2, 3, 4, 5] | k = 4 | x = -1 | khoảng cách: [2, 2, 3, 4, 5, 6] | kỳ vọng: [1, 1, 2, 3]


## 3. Kiến thức nền

### 3.1. Khoảng cách `abs(v - x)` và luật hòa

`abs(n)` trả về giá trị tuyệt đối. "Gần `x`" nghĩa là `abs(v - x)` nhỏ. Để diễn đạt luật hòa (hòa thì lấy số nhỏ hơn) ta dùng **khóa là tuple** `(abs(v - x), v)`: Python so sánh tuple theo từng thành phần, nên trước hết so khoảng cách, nếu bằng nhau mới so giá trị `v`.

In [2]:
x = 3
print(abs(1 - x), abs(5 - x), abs(3 - x))
print((2, 1) < (2, 5))          # cùng khoảng cách 2, số 1 nhỏ hơn -> 1 "gần hơn" 5
print((1, 4) < (2, 1))          # khoảng cách nhỏ hơn thắng

2 2 0
True
True


### 3.2. `sorted(..., key=...)`

`sorted(iterable, key=f)` sắp xếp theo giá trị `f(phần tử)` thay vì theo chính phần tử. Sắp xếp của Python **ổn định** (stable): hai phần tử có cùng khóa giữ nguyên thứ tự ban đầu. Ở đây ta đưa luôn luật hòa vào khóa nên không cần dựa vào tính ổn định.

In [3]:
arr = [1, 2, 3, 4, 5]
x = 3
by_dist = sorted(arr, key=lambda v: (abs(v - x), v))
print(by_dist)               # [3, 2, 4, 1, 5]
print(by_dist[:4])           # 4 phần tử gần nhất (chưa đúng thứ tự tăng dần)
print(sorted(by_dist[:4]))   # sắp xếp lại theo giá trị

[3, 2, 4, 1, 5]
[3, 2, 4, 1]
[1, 2, 3, 4]


### 3.3. Cắt lát `arr[l:r]`

`arr[l:r]` lấy các phần tử có chỉ số từ `l` đến `r - 1` (bỏ `r`), và độ dài kết quả là `r - l`. Đây là cách trả về **cửa sổ** liên tiếp `k` phần tử: `arr[l : l + k]`.

In [4]:
arr = [10, 20, 30, 40, 50]
print(arr[1:4], len(arr[1:4]))
k, l = 2, 3
print(arr[l:l + k])

[20, 30, 40] 3
[40, 50]


### 3.4. `heapq` — heap nhỏ nhất và `nsmallest`

`heapq` quản lý một danh sách sao cho `heap[0]` luôn là phần tử **nhỏ nhất**. `heappush` thêm, `heappop` lấy nhỏ nhất, `heapreplace(h, v)` lấy nhỏ nhất ra và đẩy `v` vào trong một thao tác. Để giữ "`k` phần tử tốt nhất" bằng heap kích thước `k`, ta thường đổi dấu khóa để heap[0] là phần tử **tệ nhất** trong `k` phần tử đang giữ. `heapq.nsmallest(k, iterable, key=f)` trả về `k` phần tử có khóa nhỏ nhất.

In [5]:
h = []
for v in [5, 1, 4, 2, 3]:
    heapq.heappush(h, v)
print(h[0])                                  # 1: nhỏ nhất
print(heapq.heapreplace(h, 9), h[0])         # lấy 1 ra, đẩy 9 vào; nhỏ nhất mới là 2
print(heapq.nsmallest(3, [7, 2, 9, 4, 1]))
print(heapq.nsmallest(2, ["bb", "a", "ccc"], key=len))

1
1 2
[1, 2, 4]
['a', 'bb']


### 3.5. `bisect_left` — vị trí chèn trong mảng đã sắp xếp

`bisect_left(arr, x)` trả về chỉ số `i` nhỏ nhất sao cho `arr[i] >= x`, tức vị trí mà `x` được chèn để mảng vẫn tăng dần (trước các phần tử bằng `x`). Mất `O(log n)`.

In [6]:
arr = [1, 3, 3, 5, 8]
for x in (0, 3, 4, 9):
    print(f"bisect_left(arr, {x}) =", bisect_left(arr, x))

bisect_left(arr, 0) = 0
bisect_left(arr, 3) = 1
bisect_left(arr, 4) = 3
bisect_left(arr, 9) = 5


### 3.6. Khuôn mẫu binary search `while left < right`

Với bài tìm **vị trí đầu tiên thỏa một điều kiện đơn điệu** (sai ... sai, đúng ... đúng), khuôn gọn nhất là:

```
left, right = 0, n          # đáp án chắc chắn trong [left, right]
while left < right:
    mid = (left + right) >> 1
    if điều_kiện(mid):      # mid có thể là đáp án
        right = mid
    else:
        left = mid + 1
return left
```

`(left + right) >> 1` bằng `(left + right) // 2`. Khi `left < right`, `mid < right` luôn đúng.

In [7]:
# Ví dụ nhỏ: vị trí đầu tiên có arr[i] >= 5 (mảng tăng dần)
a = [1, 3, 4, 5, 7, 9]
left, right = 0, len(a)
while left < right:
    mid = (left + right) >> 1
    print(f"left={left} right={right} mid={mid} a[mid]={a[mid]}", "-> right = mid" if a[mid] >= 5 else "-> left = mid + 1")
    if a[mid] >= 5:
        right = mid
    else:
        left = mid + 1
print("vị trí đầu tiên >= 5:", left)

left=0 right=6 mid=3 a[mid]=5 -> right = mid
left=0 right=3 mid=1 a[mid]=3 -> left = mid + 1
left=2 right=3 mid=2 a[mid]=4 -> left = mid + 1
vị trí đầu tiên >= 5: 3


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `arr = [1,2,3,4,5]`, `x = 3`. Xếp các phần tử theo **khoảng cách đến `x`** (hòa thì số nhỏ hơn trước):

| giá trị | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| khoảng cách | 2 | 1 | 0 | 1 | 2 |
| hạng gần nhất | 4 | 2 | 1 | 3 | 5 |

Lấy `k = 4` phần tử gần nhất là hạng 1..4: `3, 2, 4, 1` → sắp xếp lại: `[1,2,3,4]`.

**Quan sát quan trọng:** các phần tử được chọn `{1,2,3,4}` nằm **liền nhau** trong mảng. Vì sao? Mảng tăng dần nên khoảng cách `|arr[i] - x|` **giảm dần** khi đi từ trái tới vị trí của `x`, rồi **tăng dần** sau đó (giống chữ V). Tập `k` giá trị có khoảng cách nhỏ nhất của một hàm dạng chữ V luôn là một đoạn liên tục. Hãy kiểm chứng bằng thực nghiệm:

In [8]:
def closest_by_sort(arr, k, x):
    return sorted(sorted(arr, key=lambda v: (abs(v - x), v))[:k])

rng = random.Random(1)
for _ in range(2000):
    n = rng.randint(1, 12)
    arr = sorted(rng.randint(-6, 6) for _ in range(n))
    k = rng.randint(1, n)
    x = rng.randint(-8, 8)
    ans = closest_by_sort(arr, k, x)
    # tồn tại cửa sổ liên tiếp dài k trùng đúng với đáp án
    assert any(arr[l:l + k] == ans for l in range(n - k + 1)), (arr, k, x, ans)
print("2000 trường hợp ngẫu nhiên: đáp án luôn là một cửa sổ liên tiếp dài k")

2000 trường hợp ngẫu nhiên: đáp án luôn là một cửa sổ liên tiếp dài k


Vậy bài toán trở thành: **chọn chỉ số bắt đầu `l` của cửa sổ `arr[l : l + k]`**, `l` thuộc `0..n-k`. Cửa sổ nào tốt nhất? Mục 6 trả lời bằng cách so hai cửa sổ kề nhau.

## 5. Cách 1 — Brute Force: sắp xếp theo khoảng cách

**Ý tưởng.** Làm đúng như đề: sắp xếp toàn bộ phần tử theo khóa `(|v - x|, v)`, lấy `k` phần tử đầu (là `k` phần tử gần nhất), rồi **sắp xếp lại theo giá trị** để trả về tăng dần.

In [9]:
def closest_sort(arr, k, x):
    ordered = sorted(arr, key=lambda v: (abs(v - x), v))   # gần nhất trước
    return sorted(ordered[:k])                             # lấy k đầu, rồi sắp xếp theo giá trị


for (arr, k, x), expected in examples:
    got = closest_sort(arr, k, x)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 4] (kỳ vọng [1, 2, 3, 4] )
[1, 1, 2, 3] (kỳ vọng [1, 1, 2, 3] )


**Độ phức tạp:** Time `O(n log n)` (sắp xếp `n` phần tử, cộng `O(k log k)` để sắp xếp lại). Space `O(n)`.

**Nút thắt:** ta **không dùng** tính chất "mảng đã sắp xếp" và "đáp án là cửa sổ liên tiếp". Sắp xếp lại toàn bộ là phí phạm: mảng đã gần như có thứ tự sẵn. Các cách sau khai thác điều này.

## 6. Ý tưởng thuật toán / bước đột phá

**Đáp án là cửa sổ.** Gọi cửa sổ bắt đầu ở `l` là `W(l) = arr[l : l + k]`, `l` từ `0` đến `n - k`. So hai cửa sổ kề nhau `W(l)` và `W(l + 1)`: chúng chung `k - 1` phần tử, `W(l)` có thêm `arr[l]` và `W(l + 1)` có thêm `arr[l + k]`. Vậy:

- Giữ `W(l)` (bỏ `arr[l + k]`) nếu `arr[l]` **gần `x` hơn hoặc bằng** `arr[l + k]` (hòa thì chọn số nhỏ hơn, mà `arr[l] <= arr[l + k]`).
- Ngược lại chuyển sang `W(l + 1)`.

**So sánh không cần `abs`.** Vì `arr[l] <= arr[l + k]`, điều kiện "`arr[l]` gần `x` hơn hoặc bằng `arr[l + k]`" tương đương:

```
x - arr[l]  <=  arr[l + k] - x
```

(kiểm tra ba trường hợp: `x` bên trái cả hai, `x` giữa hai số, `x` bên phải cả hai — mục 10 chứng minh).

**Tính đơn điệu.** Khi `l` tăng, `arr[l]` không giảm nên `x - arr[l]` **không tăng**, còn `arr[l + k]` không giảm nên `arr[l + k] - x` **không giảm**. Vế trái đi xuống, vế phải đi lên, nên điều kiện `P(l): x - arr[l] <= arr[l + k] - x` **một khi đã đúng thì đúng mãi**: chuỗi `P(0), P(1), ..., P(n-k-1)` có dạng `sai ... sai, đúng ... đúng`.

**Kết luận.** Cửa sổ tốt nhất bắt đầu ở **`l` đầu tiên có `P(l)` đúng** (vì trước đó `W(l)` thua `W(l+1)`, từ đó trở đi `W(l)` thắng `W(l+1)`), hoặc `l = n - k` nếu không có `l` nào thỏa. Đó là bài **binary search tìm vị trí đầu tiên thỏa điều kiện** với `left = 0`, `right = n - k`.

## 7. Các cách tiếp theo

Phần này có thêm 4 cách nữa (tổng 5 cách): heap kích thước `k`, hai con trỏ thu hẹp, binary search điểm bắt đầu cửa sổ, và `bisect` rồi nở ra hai phía.

### 7.1. Cách 2 — Heap kích thước `k` (giữ `k` phần tử gần nhất đang biết)

**Giải thích từng đoạn.** Duyệt từng phần tử `v`. Giữ một **max-heap theo "độ tệ"** gồm tối đa `k` phần tử tốt nhất hiện có; phần tử ở đỉnh là phần tử **tệ nhất** trong `k` phần tử đang giữ. Với Python (`heapq` là min-heap) ta lưu khóa đã đổi dấu `(-|v - x|, -v)`: khóa nhỏ nhất của heap tương ứng với phần tử **xa nhất** (và, khi hòa, **lớn nhất**). Khi `v` mới có khóa gốc nhỏ hơn đỉnh thì thay thế đỉnh. Cuối cùng sắp xếp `k` phần tử còn lại.

In [10]:
def closest_heap(arr, k, x):
    heap = []                                     # chứa (-khoảng cách, -giá trị)
    for v in arr:
        item = (-abs(v - x), -v)                  # item lớn hơn <=> phần tử gần hơn
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:                      # gần hơn phần tử tệ nhất đang giữ
            heapq.heapreplace(heap, item)
    return sorted(-neg_v for _, neg_v in heap)


for (arr, k, x), expected in examples:
    got = closest_heap(arr, k, x)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 4] (kỳ vọng [1, 2, 3, 4] )
[1, 1, 2, 3] (kỳ vọng [1, 1, 2, 3] )


In [11]:
class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        heap = []
        for v in arr:
            item = (-abs(v - x), -v)
            if len(heap) < k:
                heapq.heappush(heap, item)
            elif item > heap[0]:
                heapq.heapreplace(heap, item)
        return sorted(-neg_v for _, neg_v in heap)

print(Solution().findClosestElements([1, 2, 3, 4, 5], 4, 3))

[1, 2, 3, 4]


**Độ phức tạp:** Time `O(n log k)` (mỗi phần tử có thể đẩy vào/ra heap kích thước `k`) cộng `O(k log k)` sắp xếp kết quả. Space `O(k)`. Đây là cách tổng quát dùng được cả khi mảng **chưa** sắp xếp (xem biến thể ở mục 15).

### 7.2. Cách 3 — Hai con trỏ: thu hẹp từ hai đầu

**Giải thích từng đoạn.** Bắt đầu với cửa sổ **cả mảng** `arr[l:r]` với `l = 0`, `r = n` (nửa mở). Khi độ dài `r - l > k`, ta phải bỏ bớt một đầu. Phần tử ở hai đầu là `arr[l]` và `arr[r - 1]`. Đầu nào **xa `x` hơn** thì chắc chắn không thuộc đáp án (vì cửa sổ cuối vẫn còn thừa chỗ), nên loại nó. Hòa thì loại đầu **phải** (số lớn hơn, theo luật hòa). Dùng điều kiện đã rút gọn: `x - arr[l] <= arr[r - 1] - x` thì `arr[l]` gần hơn hoặc bằng → bỏ `arr[r - 1]` (`r -= 1`), ngược lại bỏ `arr[l]` (`l += 1`).

In [12]:
def closest_two_pointers(arr, k, x):
    l, r = 0, len(arr)
    while r - l > k:
        if x - arr[l] <= arr[r - 1] - x:
            r -= 1                # đầu phải xa hơn hoặc hòa -> bỏ đầu phải
        else:
            l += 1                # đầu trái xa hơn -> bỏ đầu trái
    return arr[l:r]


for (arr, k, x), expected in examples:
    got = closest_two_pointers(arr, k, x)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 4] (kỳ vọng [1, 2, 3, 4] )
[1, 1, 2, 3] (kỳ vọng [1, 1, 2, 3] )


In [13]:
class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        l, r = 0, len(arr)
        while r - l > k:
            if x - arr[l] <= arr[r - 1] - x:
                r -= 1
            else:
                l += 1
        return arr[l:r]

print(Solution().findClosestElements([1, 1, 2, 3, 4, 5], 4, -1))

[1, 1, 2, 3]


**Độ phức tạp:** Time `O(n - k)` (mỗi vòng bỏ đúng một phần tử), cộng `O(k)` để cắt lát kết quả. Space `O(k)` cho kết quả (con trỏ chỉ `O(1)`). Kết quả đã sắp xếp sẵn vì là lát cắt của mảng tăng dần.

### 7.3. Cách 4 — Binary search điểm bắt đầu của cửa sổ (cách nên nộp)

**Giải thích từng đoạn.** Theo mục 6, ta tìm `l` đầu tiên trong `[0, n - k]` thỏa `P(l): x - arr[l] <= arr[l + k] - x`. Đặt `left = 0`, `right = n - k` (đáp án chắc chắn trong `[left, right]`). Tại `mid`: nếu `P(mid)` đúng thì cửa sổ `mid` thắng cửa sổ `mid + 1`, đáp án nằm ở `mid` hoặc bên trái → `right = mid`; ngược lại `left = mid + 1`. Vì `mid < right <= n - k` nên `mid + k <= n - 1`: không bao giờ vượt mảng.

In [14]:
def closest_binary_search(arr, k, x):
    left, right = 0, len(arr) - k
    while left < right:
        mid = (left + right) >> 1
        if x - arr[mid] <= arr[mid + k] - x:
            right = mid
        else:
            left = mid + 1
    return arr[left:left + k]


for (arr, k, x), expected in examples:
    got = closest_binary_search(arr, k, x)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 4] (kỳ vọng [1, 2, 3, 4] )
[1, 1, 2, 3] (kỳ vọng [1, 1, 2, 3] )


In [15]:
class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        left, right = 0, len(arr) - k
        while left < right:
            mid = (left + right) >> 1
            if x - arr[mid] <= arr[mid + k] - x:
                right = mid
            else:
                left = mid + 1
        return arr[left:left + k]

print(Solution().findClosestElements([1, 2, 3, 4, 5], 4, 3))

[1, 2, 3, 4]


**Độ phức tạp:** Time `O(log(n - k))` cho phần tìm kiếm cộng `O(k)` để cắt lát kết quả. Space `O(k)` cho kết quả (không tính lát cắt thì `O(1)`).

### 7.4. Cách 5 — `bisect` rồi nở ra hai phía

**Giải thích từng đoạn.** Dùng `bisect_left` tìm vị trí `i` đầu tiên có `arr[i] >= x`. Đặt hai con trỏ `lo = i - 1` (phần tử cuối cùng `< x`) và `hi = i` (phần tử đầu tiên `>= x`). Lặp `k` lần: chọn một trong hai phần tử ở hai biên — phần tử gần `x` hơn (hòa thì lấy phía trái, số nhỏ hơn) — rồi dịch con trỏ tương ứng. Cần xử lý khi một phía đã hết mảng (`lo < 0` hoặc `hi >= n`). Sau `k` bước, cửa sổ là `arr[lo + 1 : hi]`.

In [16]:
def closest_expand(arr, k, x):
    n = len(arr)
    hi = bisect_left(arr, x)       # arr[hi] là phần tử đầu tiên >= x
    lo = hi - 1                    # arr[lo] là phần tử cuối cùng < x
    for _ in range(k):
        if lo < 0:
            hi += 1                # hết phía trái: buộc lấy phía phải
        elif hi >= n:
            lo -= 1                # hết phía phải: buộc lấy phía trái
        elif x - arr[lo] <= arr[hi] - x:
            lo -= 1                # trái gần hơn hoặc hòa -> lấy trái
        else:
            hi += 1
    return arr[lo + 1:hi]


for (arr, k, x), expected in examples:
    got = closest_expand(arr, k, x)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 4] (kỳ vọng [1, 2, 3, 4] )
[1, 1, 2, 3] (kỳ vọng [1, 1, 2, 3] )


In [17]:
class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        n = len(arr)
        hi = bisect_left(arr, x)
        lo = hi - 1
        for _ in range(k):
            if lo < 0:
                hi += 1
            elif hi >= n:
                lo -= 1
            elif x - arr[lo] <= arr[hi] - x:
                lo -= 1
            else:
                hi += 1
        return arr[lo + 1:hi]

print(Solution().findClosestElements([1, 1, 2, 3, 4, 5], 4, -1))

[1, 1, 2, 3]


**Độ phức tạp:** Time `O(log n + k)` (một lần `bisect`, rồi `k` bước nở ra). Space `O(k)` cho kết quả. Rất gọn khi `k` nhỏ so với `n`, và nó là hình ảnh trực quan nhất của "mở rộng từ vị trí của `x`".

## 8. TRACE từng bước

Trace cách binary search (cách 4) với ví dụ 1: `arr = [1,2,3,4,5]`, `k = 4`, `x = 3`, `n = 5`, kỳ vọng `[1,2,3,4]`. `left = 0`, `right = n - k = 1`.

| bước | left | right | mid | `x - arr[mid]` | `arr[mid+k] - x` | điều kiện `<=` | hành động |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 1 | 0 | 3 - 1 = 2 | arr[4] - 3 = 2 | đúng (hòa) | right = 0 |
| dừng | 0 | 0 | | | | | trả `arr[0:4] = [1,2,3,4]` |

Nhận xét: hòa (`2 <= 2`) đưa `right = mid`, tức **ưu tiên cửa sổ bên trái** (giữ số nhỏ hơn).

In [18]:
def trace_binary_search(arr, k, x):
    left, right = 0, len(arr) - k
    print(f"arr={arr}, k={k}, x={x}, left={left}, right={right}")
    step = 0
    while left < right:
        step += 1
        mid = (left + right) >> 1
        lhs, rhs = x - arr[mid], arr[mid + k] - x
        ok = lhs <= rhs
        action = f"right = {mid}" if ok else f"left = {mid + 1}"
        print(f"bước {step}: left={left} right={right} mid={mid} | x-arr[mid]={lhs} arr[mid+k]-x={rhs} -> {'đúng' if ok else 'sai'} -> {action}")
        if ok:
            right = mid
        else:
            left = mid + 1
    print(f"dừng: left=right={left} -> trả arr[{left}:{left + k}] = {arr[left:left + k]}")
    return arr[left:left + k]

assert trace_binary_search([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]

arr=[1, 2, 3, 4, 5], k=4, x=3, left=0, right=1
bước 1: left=0 right=1 mid=0 | x-arr[mid]=2 arr[mid+k]-x=2 -> đúng -> right = 0
dừng: left=right=0 -> trả arr[0:4] = [1, 2, 3, 4]


Trace ví dụ 2: `arr = [1,1,2,3,4,5]`, `k = 4`, `x = -1`, `n = 6`, `right = 2`.

| bước | left | right | mid | `x - arr[mid]` | `arr[mid+k] - x` | điều kiện | hành động |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 2 | 1 | -1 - 1 = -2 | arr[5] + 1 = 6 | đúng | right = 1 |
| 2 | 0 | 1 | 0 | -1 - 1 = -2 | arr[4] + 1 = 5 | đúng | right = 0 |
| dừng | 0 | 0 | | | | | trả `arr[0:4] = [1,1,2,3]` |

In [19]:
assert trace_binary_search([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
print()
# x nằm bên phải mọi phần tử: cửa sổ phải đi tới cuối mảng
assert trace_binary_search([1, 2, 3, 4, 5], 2, 100) == [4, 5]
print()
# một ca có hòa ở giữa: mảng [1, 3], k = 1, x = 2 -> chọn 1 (nhỏ hơn)
assert trace_binary_search([1, 3], 1, 2) == [1]

arr=[1, 1, 2, 3, 4, 5], k=4, x=-1, left=0, right=2
bước 1: left=0 right=2 mid=1 | x-arr[mid]=-2 arr[mid+k]-x=6 -> đúng -> right = 1
bước 2: left=0 right=1 mid=0 | x-arr[mid]=-2 arr[mid+k]-x=5 -> đúng -> right = 0
dừng: left=right=0 -> trả arr[0:4] = [1, 1, 2, 3]

arr=[1, 2, 3, 4, 5], k=2, x=100, left=0, right=3
bước 1: left=0 right=3 mid=1 | x-arr[mid]=98 arr[mid+k]-x=-96 -> sai -> left = 2
bước 2: left=2 right=3 mid=2 | x-arr[mid]=97 arr[mid+k]-x=-95 -> sai -> left = 3
dừng: left=right=3 -> trả arr[3:5] = [4, 5]

arr=[1, 3], k=1, x=2, left=0, right=1
bước 1: left=0 right=1 mid=0 | x-arr[mid]=1 arr[mid+k]-x=1 -> đúng -> right = 0
dừng: left=right=0 -> trả arr[0:1] = [1]


Trace cách hai con trỏ trên ví dụ 1 để thấy cửa sổ co lại:

| bước | l | r | độ dài | `x - arr[l]` | `arr[r-1] - x` | hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 5 | 5 | 2 | 2 | `2 <= 2` đúng: r = 4 |
| dừng | 0 | 4 | 4 | | | trả `arr[0:4]` |

In [20]:
def trace_two_pointers(arr, k, x):
    l, r = 0, len(arr)
    step = 0
    while r - l > k:
        step += 1
        lhs, rhs = x - arr[l], arr[r - 1] - x
        if lhs <= rhs:
            print(f"bước {step}: l={l} r={r} cửa sổ={arr[l:r]} | x-arr[l]={lhs} <= arr[r-1]-x={rhs} -> bỏ đầu phải {arr[r-1]}")
            r -= 1
        else:
            print(f"bước {step}: l={l} r={r} cửa sổ={arr[l:r]} | x-arr[l]={lhs} > arr[r-1]-x={rhs} -> bỏ đầu trái {arr[l]}")
            l += 1
    print(f"dừng: cửa sổ = {arr[l:r]}")
    return arr[l:r]

assert trace_two_pointers([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
print()
assert trace_two_pointers([0, 0, 1, 2, 3, 3, 4, 7, 8, 8], 3, 5) == [3, 3, 4]

bước 1: l=0 r=5 cửa sổ=[1, 2, 3, 4, 5] | x-arr[l]=2 <= arr[r-1]-x=2 -> bỏ đầu phải 5
dừng: cửa sổ = [1, 2, 3, 4]

bước 1: l=0 r=10 cửa sổ=[0, 0, 1, 2, 3, 3, 4, 7, 8, 8] | x-arr[l]=5 > arr[r-1]-x=3 -> bỏ đầu trái 0
bước 2: l=1 r=10 cửa sổ=[0, 1, 2, 3, 3, 4, 7, 8, 8] | x-arr[l]=5 > arr[r-1]-x=3 -> bỏ đầu trái 0
bước 3: l=2 r=10 cửa sổ=[1, 2, 3, 3, 4, 7, 8, 8] | x-arr[l]=4 > arr[r-1]-x=3 -> bỏ đầu trái 1
bước 4: l=3 r=10 cửa sổ=[2, 3, 3, 4, 7, 8, 8] | x-arr[l]=3 <= arr[r-1]-x=3 -> bỏ đầu phải 8
bước 5: l=3 r=9 cửa sổ=[2, 3, 3, 4, 7, 8] | x-arr[l]=3 <= arr[r-1]-x=3 -> bỏ đầu phải 8
bước 6: l=3 r=8 cửa sổ=[2, 3, 3, 4, 7] | x-arr[l]=3 > arr[r-1]-x=2 -> bỏ đầu trái 2
bước 7: l=4 r=8 cửa sổ=[3, 3, 4, 7] | x-arr[l]=2 <= arr[r-1]-x=2 -> bỏ đầu phải 7
dừng: cửa sổ = [3, 3, 4]


## 9. Code nộp LeetCode

Lời giải gọn và nhanh nhất (Cách 4 - binary search điểm bắt đầu cửa sổ):

In [21]:
from typing import List


class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        left, right = 0, len(arr) - k          # l thuộc [0, n - k]
        while left < right:
            mid = (left + right) >> 1
            if x - arr[mid] <= arr[mid + k] - x:   # arr[mid] gần hơn hoặc hòa -> cửa sổ không cần dịch phải
                right = mid
            else:
                left = mid + 1
        return arr[left:left + k]


sol = Solution()
for (arr, k, x), expected in examples:
    got = sol.findClosestElements(list(arr), k, x)
    print(got)
    assert got == expected

[1, 2, 3, 4]
[1, 1, 2, 3]


## 10. Vì sao đúng

**Bổ đề 1 — đáp án là cửa sổ liên tiếp.** Giả sử đáp án chứa `arr[i]` và `arr[j]` (`i < j`) nhưng bỏ `arr[m]` với `i < m < j`, nên `arr[i] <= arr[m] <= arr[j]`. Xét vị trí của `x`: nếu `x <= arr[m]` thì `arr[m]` gần `x` hơn hoặc bằng `arr[j]` (cả hai bên phải `x`, `arr[m]` nhỏ hơn); nếu `x >= arr[m]` thì `arr[m]` gần `x` hơn hoặc bằng `arr[i]`. Trong cả hai trường hợp, đổi phần tử tương ứng (`arr[j]` hoặc `arr[i]`) lấy `arr[m]` không làm đáp án tệ đi (nếu khoảng cách bằng nhau thì hai giá trị bằng nhau nên đổi cũng không thay đổi danh sách giá trị). Vậy luôn có đáp án tối ưu liên tục (đã kiểm chứng thực nghiệm ở mục 4).

**Bổ đề 2 — so sánh hai cửa sổ kề nhau.** `W(l)` hơn `W(l + 1)` khi `arr[l]` "gần hơn hoặc hòa" `arr[l + k]` (hòa → số nhỏ hơn thắng; mà `arr[l] <= arr[l + k]`). Do `arr[l] <= arr[l + k]` ta có ba trường hợp:

- `x <= arr[l]`: cả hai nằm bên phải `x`, `arr[l]` gần hơn hoặc bằng. Điều kiện `x - arr[l] <= 0 <= arr[l + k] - x` đúng.
- `arr[l] < x < arr[l + k]`: `x - arr[l]` và `arr[l + k] - x` đều dương, chính là hai khoảng cách; so sánh chúng đúng bằng điều kiện.
- `x >= arr[l + k]`: cả hai nằm bên trái `x`, `arr[l + k]` gần hơn hoặc bằng. Khi đó `arr[l + k] - x <= 0 <= x - arr[l]` nên điều kiện chỉ đúng khi cả hai vế bằng `0`, tức `x = arr[l] = arr[l + k]` (hai số bằng nhau, hai cửa sổ cho cùng giá trị nên chọn bên nào cũng đúng); các trường hợp còn lại điều kiện sai và ta dịch sang phải, đúng như mong muốn.

**Bổ đề 3 — đơn điệu.** Đã chứng minh ở mục 6: `x - arr[l]` không tăng và `arr[l + k] - x` không giảm khi `l` tăng; do đó `P(l)` đúng thì `P(l + 1)` đúng.

**Kết luận.** Gọi `l*` là `l` đầu tiên có `P(l)` đúng (hoặc `n - k` nếu không có). Với `l < l*`, `W(l)` thua `W(l + 1)`; với `l >= l*` (và `l + 1 <= n - k`), `W(l)` không thua `W(l + 1)`. Chuỗi `W(0) < W(1) < ... < W(l*) >= W(l*+1) >= ...` nên `W(l*)` là tốt nhất; "tốt nhất" so theo thứ tự "gần hơn" phần tử-theo-phần-tử, nhất quán vì ba bổ đề trên. Binary search `while left < right` với `right = mid` / `left = mid + 1` tìm đúng `l*`, và chỉ đọc `arr[mid + k]` với `mid < n - k` nên không vượt mảng.

In [22]:
# Kiểm chứng Bổ đề 3: P(l) đơn điệu (sai...sai, đúng...đúng) trên nhiều mảng ngẫu nhiên
rng = random.Random(10)
for _ in range(3000):
    n = rng.randint(1, 15)
    arr = sorted(rng.randint(-8, 8) for _ in range(n))
    k = rng.randint(1, n)
    x = rng.randint(-10, 10)
    P = [x - arr[l] <= arr[l + k] - x for l in range(n - k)]
    assert P == sorted(P), (arr, k, x, P)       # False < True nên sorted(P) = sai...sai, đúng...đúng
print("Bổ đề 3 đúng trên 3000 trường hợp ngẫu nhiên")

Bổ đề 3 đúng trên 3000 trường hợp ngẫu nhiên


## 11. Time / Space Complexity

Đặt `n = len(arr)`.

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Sắp xếp theo khoảng cách | `O(n log n)` | `O(n)` |
| 2. Heap kích thước `k` | `O(n log k)` | `O(k)` |
| 3. Hai con trỏ thu hẹp | `O(n - k)` | `O(1)` (không tính kết quả) |
| 4. Binary search điểm bắt đầu | `O(log(n - k) + k)` | `O(1)` (không tính kết quả) |
| 5. `bisect` + nở ra hai phía | `O(log n + k)` | `O(1)` (không tính kết quả) |

Cách 4 là cách tốt nhất về lý thuyết: phần tìm kiếm chỉ `O(log(n - k))`, phần còn lại là sao chép `k` phần tử ra kết quả (không thể tránh vì phải trả về `k` phần tử). Với `n ≤ 10^4` mọi cách đều rất nhanh; điểm mấu chốt là hiểu cấu trúc cửa sổ.

In [23]:
rng = random.Random(658)
n = 10_000
big = sorted(rng.randint(-10_000, 10_000) for _ in range(n))
solvers_all = {
    "sort": closest_sort, "heap": closest_heap, "two_ptr": closest_two_pointers,
    "bsearch": closest_binary_search, "expand": closest_expand,
}
for k in (1, 100, 5000, n):
    x = rng.randint(-10_000, 10_000)
    results = {}
    for name, f in solvers_all.items():
        t0 = time.perf_counter()
        results[name] = f(big, k, x)
        assert time.perf_counter() - t0 < 5
    assert all(r == results["sort"] for r in results.values()), k
    print(f"k={k:5d}: kết quả dài {len(results['sort'])}, cả 5 cách khớp nhau")

k=    1: kết quả dài 1, cả 5 cách khớp nhau


k=  100: kết quả dài 100, cả 5 cách khớp nhau


k= 5000: kết quả dài 5000, cả 5 cách khớp nhau
k=10000: kết quả dài 10000, cả 5 cách khớp nhau


## 12. Bộ test

Chia 4 nhóm: (a) ví dụ đề và edge case; (b) phản ví dụ cho heuristic sai; (c) đối chiếu mọi cách với **oracle độc lập** (thử mọi tổ hợp `k` phần tử); (d) test ngẫu nhiên có seed.

### 12.1. Oracle độc lập và test cố định

Oracle: thử **mọi tổ hợp `k` chỉ số**; mỗi tổ hợp tính "bảng độ gần" = danh sách khóa `(|v - x|, v)` đã sắp xếp; chọn tổ hợp có bảng nhỏ nhất theo thứ tự từ điển. Không dùng binary search hay tính chất cửa sổ.

In [24]:
def oracle_combinations(arr, k, x):
    best_key, best_vals = None, None
    for idxs in combinations(range(len(arr)), k):
        vals = [arr[i] for i in idxs]
        key = sorted((abs(v - x), v) for v in vals)
        if best_key is None or key < best_key:
            best_key, best_vals = key, vals
    return sorted(best_vals)


solvers = {
    "sort": closest_sort,
    "heap": closest_heap,
    "two_ptr": closest_two_pointers,
    "bsearch": closest_binary_search,
    "expand": closest_expand,
    "submit": lambda arr, k, x: Solution().findClosestElements(list(arr), k, x),
}

fixed_cases = [
    (([1, 2, 3, 4, 5], 4, 3), [1, 2, 3, 4]),          # ví dụ 1: hòa 1 và 5, lấy 1
    (([1, 1, 2, 3, 4, 5], 4, -1), [1, 1, 2, 3]),      # ví dụ 2: x nhỏ hơn cả mảng
    (([1], 1, 100), [1]),                              # một phần tử
    (([1, 2, 3, 4, 5], 5, 3), [1, 2, 3, 4, 5]),        # k = n
    (([1, 2, 3, 4, 5], 1, 3), [3]),                    # k = 1, x có trong mảng
    (([1, 2, 3, 4, 5], 2, 100), [4, 5]),               # x lớn hơn cả mảng
    (([1, 3], 1, 2), [1]),                             # hòa hoàn toàn: lấy số nhỏ hơn
    (([1, 3], 2, 2), [1, 3]),
    (([1, 1, 1, 1], 2, 1), [1, 1]),                    # toàn phần tử trùng nhau
    (([0, 0, 1, 2, 3, 3, 4, 7, 8, 8], 3, 5), [3, 3, 4]),   # ca kinh điển có trùng lặp
    (([1, 10, 11, 12], 2, 9), [10, 11]),
    (([-5, -3, 0, 2, 4], 3, -1), [-3, 0, 2]),          # số âm
    (([1, 2, 3, 4, 5], 3, 3), [2, 3, 4]),
]
for (arr, k, x), expected in fixed_cases:
    assert oracle_combinations(arr, k, x) == expected, ("oracle", arr, k, x)
    for name, f in solvers.items():
        got = f(arr, k, x)
        assert got == expected, (name, arr, k, x, got, expected)
print("Test cố định: OK (", len(fixed_cases), "bộ x", len(solvers), "lời giải + oracle )")

Test cố định: OK ( 13 bộ x 6 lời giải + oracle )


### 12.2. Phản ví dụ cho các heuristic sai

- *"Tìm vị trí chèn của `x` rồi lấy cửa sổ đối xứng quanh đó"* (`k/2` bên trái, `k/2` bên phải, kẹp biên). Sai khi các giá trị phân bố lệch: `arr = [1,10,11,12]`, `x = 9`, `k = 2`: vị trí chèn là `1`, cửa sổ đối xứng là `[1,10]`, nhưng đáp án `[10,11]`.
- *"So sánh chỉ `arr[mid]` với `x`"* trong binary search để quyết định hướng, mà bỏ qua `arr[mid + k]`. Cũng sai với ví dụ trên.

In [25]:
def wrong_symmetric_window(arr, k, x):
    i = bisect_left(arr, x)
    l = max(0, min(i - k // 2, len(arr) - k))     # SAI: giả định đáp án đối xứng quanh x
    return arr[l:l + k]


for (arr, k, x) in [([1, 10, 11, 12], 2, 9), ([1, 2, 3, 100, 101], 3, 50)]:
    print("đối xứng:", wrong_symmetric_window(arr, k, x), "| đúng:", closest_sort(arr, k, x))
    assert wrong_symmetric_window(arr, k, x) != closest_sort(arr, k, x)
    assert oracle_combinations(arr, k, x) == closest_sort(arr, k, x) == Solution().findClosestElements(arr, k, x)

đối xứng: [1, 10] | đúng: [10, 11]
đối xứng: [3, 100, 101] | đúng: [1, 2, 3]


### 12.3. Đối chiếu với oracle trên mọi mảng nhỏ

Vét **mọi** mảng không giảm độ dài ≤ 6 với giá trị trong `0..4`, mọi `k`, mọi `x` trong `-1..5`.

In [26]:
from itertools import combinations_with_replacement

checked = 0
for n in range(1, 7):
    for arr in combinations_with_replacement(range(0, 5), n):
        arr = list(arr)
        for k in range(1, n + 1):
            for x in range(-1, 6):
                expected = oracle_combinations(arr, k, x)
                for name, f in solvers.items():
                    got = f(arr, k, x)
                    assert got == expected, (name, arr, k, x, got, expected)
                checked += 1
print("Vét hết mảng nhỏ:", checked, "bộ (arr, k, x), mọi cách đều khớp oracle")

Vét hết mảng nhỏ: 16170 bộ (arr, k, x), mọi cách đều khớp oracle


### 12.4. Test ngẫu nhiên có seed

Nhóm 1: cỡ nhỏ (có oracle). Nhóm 2: cỡ lớn đúng ràng buộc (`n` đến `10^4`, giá trị `[-10^4, 10^4]`), so các cách với nhau.

In [27]:
rng = random.Random(658)
for _ in range(1500):
    n = rng.randint(1, 10)
    arr = sorted(rng.randint(-10, 10) for _ in range(n))
    k = rng.randint(1, n)
    x = rng.randint(-12, 12)
    expected = oracle_combinations(arr, k, x)
    for name, f in solvers.items():
        assert f(arr, k, x) == expected, (name, arr, k, x)
print("Ngẫu nhiên cỡ nhỏ (seed 658): 1500 bộ, khớp oracle")

Ngẫu nhiên cỡ nhỏ (seed 658): 1500 bộ, khớp oracle


In [28]:
rng = random.Random(2026)
for case in range(60):
    n = rng.randint(1, 10_000)
    span = rng.choice([5, 100, 10_000])
    arr = sorted(rng.randint(-span, span) for _ in range(n))
    k = rng.randint(1, n)
    x = rng.randint(-10_000, 10_000)
    assert 1 <= k <= n and all(-10_000 <= v <= 10_000 for v in arr)
    base = closest_sort(arr, k, x)
    assert len(base) == k and base == sorted(base)
    for name, f in solvers.items():
        assert f(arr, k, x) == base, (name, n, k, x)
print("Ngẫu nhiên cỡ lớn (seed 2026): 60 bộ, mọi cách khớp nhau")

Ngẫu nhiên cỡ lớn (seed 2026): 60 bộ, mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` thay vì `<=` khi hòa

Khi `x - arr[mid] == arr[mid + k] - x` (hòa), luật đề chọn **số nhỏ hơn**, tức giữ phía trái (`right = mid`). Nếu viết `<` thì hòa bị đẩy sang phải và chọn số lớn hơn: ví dụ 1 trả `[2,3,4,5]` thay vì `[1,2,3,4]`.

In [29]:
def wrong_strict(arr, k, x):
    left, right = 0, len(arr) - k
    while left < right:
        mid = (left + right) >> 1
        if x - arr[mid] < arr[mid + k] - x:      # SAI: phải là <=
            right = mid
        else:
            left = mid + 1
    return arr[left:left + k]

print("sai:", wrong_strict([1, 2, 3, 4, 5], 4, 3), "| đúng:", Solution().findClosestElements([1, 2, 3, 4, 5], 4, 3))
assert wrong_strict([1, 2, 3, 4, 5], 4, 3) == [2, 3, 4, 5]

sai: [2, 3, 4, 5] | đúng: [1, 2, 3, 4]


### Lỗi 2 — `right = len(arr) - 1` làm `mid + k` vượt mảng

Cửa sổ bắt đầu ở `l` chỉ hợp lệ khi `l + k <= n`, tức `l <= n - k`. Đặt `right = n - 1` thì `mid` có thể lớn hơn `n - k - 1` và `arr[mid + k]` báo `IndexError`.

In [30]:
def wrong_right_bound(arr, k, x):
    left, right = 0, len(arr) - 1                 # SAI: right phải là len(arr) - k
    while left < right:
        mid = (left + right) >> 1
        if x - arr[mid] <= arr[mid + k] - x:
            right = mid
        else:
            left = mid + 1
    return arr[left:left + k]

try:
    wrong_right_bound([1, 2, 3], 2, 2)
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().findClosestElements([1, 2, 3], 2, 2))

IndexError: list index out of range
đúng: [1, 2]


### Lỗi 3 — Lấy cửa sổ đối xứng quanh vị trí chèn của `x`

Đã thấy ở mục 12.2: giả định `k/2` phần tử mỗi bên là sai khi dữ liệu lệch (một bên có các phần tử xa, bên kia gần). Đáp án phụ thuộc **khoảng cách thật**, không phải số lượng phần tử.

In [31]:
for (arr, k, x) in [([1, 10, 11, 12], 2, 9)]:
    print("đối xứng sai:", wrong_symmetric_window(arr, k, x), "| đúng:", Solution().findClosestElements(arr, k, x))

đối xứng sai: [1, 10] | đúng: [10, 11]


### Lỗi 4 — Chọn theo khoảng cách nhưng quên sắp xếp lại kết quả

Sau khi sắp xếp theo khoảng cách, `k` phần tử đầu **không còn thứ tự tăng dần**. Đề yêu cầu kết quả tăng dần nên phải `sorted(...)` lại (hoặc dùng cách cửa sổ để tự nhiên có thứ tự).

In [32]:
def wrong_no_resort(arr, k, x):
    return sorted(arr, key=lambda v: (abs(v - x), v))[:k]       # SAI: thiếu sorted() cuối

print("sai:", wrong_no_resort([1, 2, 3, 4, 5], 4, 3), "| đúng:", Solution().findClosestElements([1, 2, 3, 4, 5], 4, 3))
assert wrong_no_resort([1, 2, 3, 4, 5], 4, 3) == [3, 2, 4, 1]

sai: [3, 2, 4, 1] | đúng: [1, 2, 3, 4]


## 14. Mẹo & nhận dạng pattern

- **"`k` phần tử gần `x` nhất trong mảng đã sắp xếp"** → đáp án là **cửa sổ liên tục**: chuyển thành chọn điểm bắt đầu. Binary search trên **điểm bắt đầu** (không phải trên giá trị).
- **So sánh hai cửa sổ kề nhau** chỉ cần so hai phần tử mới/cũ ở hai đầu: `x - arr[mid]` với `arr[mid + k] - x`. Không cần `abs`.
- **Tìm vị trí đầu tiên thỏa điều kiện đơn điệu** → khuôn `while left < right` với `right = mid` / `left = mid + 1`; chú ý biên `right = n - k`.
- **Hòa → số nhỏ hơn**: dùng `<=` và ưu tiên bên trái.
- Mảng **chưa** sắp xếp, hoặc điểm 2D → heap `k` phần tử (`O(n log k)`) hoặc quickselect (xem LeetCode 973 K Closest Points to Origin).
- Nếu `k` nhỏ so với `n` thì `bisect` + nở ra hai phía rất tự nhiên; nếu `k` gần `n` thì hai con trỏ thu hẹp tự nhiên hơn.
- Các bài cùng họ: Search Insert Position (35), Find Peak Element (162), Kth Smallest Element in a Sorted Matrix (378).

## 15. Biến thể

### 15.1. Mảng chưa sắp xếp

Khi `arr` không sắp xếp, đáp án không còn là cửa sổ; dùng heap `k` phần tử (cách 2) hoặc `heapq.nsmallest` với khóa `(abs(v - x), v)`, rồi sắp xếp lại kết quả. Độ phức tạp `O(n log k)`.

In [33]:
def closest_unsorted(arr, k, x):
    return sorted(heapq.nsmallest(k, arr, key=lambda v: (abs(v - x), v)))

print(closest_unsorted([5, 1, 4, 2, 3], 4, 3))      # [1, 2, 3, 4]
print(closest_unsorted([9, 1, 7, 3, 5], 2, 6))      # [5, 7]
assert closest_unsorted([5, 1, 4, 2, 3], 4, 3) == closest_sort([1, 2, 3, 4, 5], 4, 3)
rng = random.Random(15)
for _ in range(500):
    arr = [rng.randint(-5, 5) for _ in range(rng.randint(1, 10))]
    k = rng.randint(1, len(arr))
    x = rng.randint(-6, 6)
    assert closest_unsorted(arr, k, x) == closest_sort(arr, k, x)
print("OK")

[1, 2, 3, 4]
[5, 7]
OK


### 15.2. Trả về chỉ số bắt đầu của cửa sổ

Nếu chỉ cần vị trí `l` bắt đầu (thay vì cả danh sách), trả `left`. Hữu ích khi `k` lớn và ta không muốn sao chép.

In [34]:
def closest_start_index(arr, k, x):
    left, right = 0, len(arr) - k
    while left < right:
        mid = (left + right) >> 1
        if x - arr[mid] <= arr[mid + k] - x:
            right = mid
        else:
            left = mid + 1
    return left

print(closest_start_index([1, 2, 3, 4, 5], 4, 3))      # 0
print(closest_start_index([0, 0, 1, 2, 3, 3, 4, 7, 8, 8], 3, 5))   # 4
assert closest_start_index([0, 0, 1, 2, 3, 3, 4, 7, 8, 8], 3, 5) == 4

0
4


### 15.3. Khoảng cách bình phương / điểm trong mặt phẳng

- **K Closest Points to Origin (973)**: khóa là `x*x + y*y`; dữ liệu không sắp xếp nên dùng heap hoặc quickselect.
- **Hòa thì lấy số lớn hơn**: đổi `<=` thành `<` trong binary search (đây là bài tập 5 ở mục 17).
- **Chỉ lấy các phần tử có `|v - x| <= d`**: hai lần `bisect` (`bisect_left(arr, x - d)` và `bisect_right(arr, x + d)`), cắt lát ở giữa.
- **Đếm số phần tử khác nhau** trong `k` gần nhất: dùng `set` trên kết quả hoặc `Counter`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp theo khoảng cách | sort khóa `(abs, v)`, lấy `k`, sort lại | `O(n log n)` | `O(n)` | đơn giản nhất; dùng được cả khi mảng chưa sắp xếp |
| 2. Heap `k` phần tử | giữ `k` phần tử gần nhất bằng max-heap | `O(n log k)` | `O(k)` | mảng chưa sắp xếp hoặc dữ liệu dạng luồng |
| 3. Hai con trỏ | bỏ đầu xa hơn cho đến khi còn `k` | `O(n - k)` | `O(1)` | `k` gần `n`; không cần binary search |
| 4. Binary search điểm bắt đầu | tìm `l` đầu tiên có `x - arr[l] <= arr[l+k] - x` | `O(log(n-k) + k)` | `O(1)` | **cách nên nộp**; nhanh nhất |
| 5. `bisect` + nở hai phía | vị trí chèn của `x`, mở rộng `k` lần | `O(log n + k)` | `O(1)` | `k` nhỏ; trực quan |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
arr = [1, 3, 6]
x = 4
print([abs(v - x) for v in arr])
print(sorted(arr, key=lambda v: (abs(v - x), v)))
print(arr[1:3])
```

In [35]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `arr = [1,2,4,7,8,9]`, `k = 3`, `x = 6`: tính khoảng cách của từng phần tử, xác định `3` phần tử gần nhất (chú ý hòa), và cho biết cửa sổ bắt đầu ở chỉ số nào cùng đáp án.

In [36]:
# Bài 2: điền kết quả tính tay
hand_start = None     # chỉ số bắt đầu cửa sổ
hand_answer = None    # đáp án (danh sách)

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành binary search điểm bắt đầu cửa sổ.

In [37]:
def exercise_3_closest(arr: List[int], k: int, x: int) -> List[int]:
    # TODO 1: left = 0, right = len(arr) - k
    left, right = None, None
    # TODO 2: while left < right: mid = trung điểm làm tròn xuống
    #     TODO 3: nếu x - arr[mid] <= arr[mid + k] - x thì right = mid, ngược lại left = mid + 1
    # TODO 4: trả về cửa sổ arr[left:left + k]
    return None


# assert exercise_3_closest([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
# assert exercise_3_closest([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
# assert exercise_3_closest([1, 3], 1, 2) == [1]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 3, hãy tự viết `exercise_4_two_pointers(arr, k, x)`: thu hẹp từ hai đầu cho đến khi còn đúng `k` phần tử.

In [38]:
def exercise_4_two_pointers(arr: List[int], k: int, x: int) -> List[int]:
    # TODO
    pass


# assert exercise_4_two_pointers([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
# assert exercise_4_two_pointers([0, 0, 1, 2, 3, 3, 4, 7, 8, 8], 3, 5) == [3, 3, 4]
# assert exercise_4_two_pointers([1, 2, 3], 3, 100) == [1, 2, 3]

### Bài 5 — Mini challenge: hòa thì lấy số **lớn** hơn (khó)

Đổi luật: nếu `|a - x| == |b - x|` thì `a` gần hơn khi `a > b` (số lớn hơn thắng). Viết `exercise_5_tie_larger(arr, k, x)` (mảng đã sắp xếp) trả về `k` phần tử gần nhất, tăng dần, theo luật mới.

```python
exercise_5_tie_larger([1, 2, 3, 4, 5], 4, 3) -> [2, 3, 4, 5]    # hòa 1 và 5: lấy 5
exercise_5_tie_larger([1, 3], 1, 2) -> [3]
exercise_5_tie_larger([1, 1, 2, 3, 4, 5], 4, -1) -> [1, 1, 2, 3]
```

Gợi ý: nghĩ xem điều kiện "đi sang phải" đổi thế nào khi hòa; viết oracle bằng sắp xếp với khóa `(abs(v - x), -v)` để kiểm tra.

In [39]:
def exercise_5_tie_larger(arr: List[int], k: int, x: int) -> List[int]:
    # TODO
    pass


# assert exercise_5_tie_larger([1, 2, 3, 4, 5], 4, 3) == [2, 3, 4, 5]
# assert exercise_5_tie_larger([1, 3], 1, 2) == [3]
# assert exercise_5_tie_larger([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `abs(v - 4)` cho từng phần tử; khóa `(khoảng cách, giá trị)`.
- Mức 2: khoảng cách là `3, 1, 2`; sắp xếp theo khoảng cách tăng dần.
- Mức 3: dòng 1 `[3, 1, 2]`; dòng 2 `[3, 6, 1]`; dòng 3 `[3, 6]`.

**Bài 2**
- Mức 1: khoảng cách với `x = 6` lần lượt `5, 4, 2, 1, 2, 3`.
- Mức 2: gần nhất là `7` (1); kế đến `4` và `8` cùng cách 2 (hòa); cần chọn đủ 3 phần tử.
- Mức 3: lấy `7, 4, 8` (cả hai số hòa đều vào vì `k = 3`): đáp án `[4, 7, 8]`, cửa sổ bắt đầu ở chỉ số `2`.

**Bài 3**
- Mức 1: giống khuôn mẫu mục 3.6 với `right = len(arr) - k`.
- Mức 2: `mid = (left + right) >> 1`; điều kiện `x - arr[mid] <= arr[mid + k] - x`.
- Mức 3: đúng thì `right = mid`, sai thì `left = mid + 1`; trả `arr[left:left + k]`.

**Bài 4**
- Mức 1: `l, r = 0, len(arr)`; lặp khi `r - l > k`.
- Mức 2: so `x - arr[l]` với `arr[r - 1] - x`.
- Mức 3: `<=` thì `r -= 1`, ngược lại `l += 1`; trả `arr[l:r]`.

**Bài 5**
- Mức 1: khi hòa, cửa sổ nên dịch về phía số lớn hơn (phải).
- Mức 2: trong binary search đổi điều kiện `<=` thành `<`.
- Mức 3: `if x - arr[mid] < arr[mid + k] - x: right = mid else: left = mid + 1`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
arr = [1, 3, 6]
x = 4
print([abs(v - x) for v in arr])
print(sorted(arr, key=lambda v: (abs(v - x), v)))
print(arr[1:3])

assert [abs(v - x) for v in arr] == [3, 1, 2]
assert sorted(arr, key=lambda v: (abs(v - x), v)) == [3, 6, 1]
assert arr[1:3] == [3, 6]
print("Bài 1 OK")

[3, 1, 2]
[3, 6, 1]
[3, 6]
Bài 1 OK


### Đáp án Bài 2

Khoảng cách: `5, 4, 2, 1, 2, 3`. Ba phần tử gần nhất: `7` (1), rồi `4` và `8` (cùng 2). Đáp án `[4, 7, 8]`, cửa sổ bắt đầu ở chỉ số `2`.

In [41]:
arr2, k2, x2 = [1, 2, 4, 7, 8, 9], 3, 6
print([abs(v - x2) for v in arr2])
ans2 = Solution().findClosestElements(arr2, k2, x2)
start2 = closest_start_index(arr2, k2, x2)
print(ans2, start2)

assert [abs(v - x2) for v in arr2] == [5, 4, 2, 1, 2, 3]
assert ans2 == [4, 7, 8] and start2 == 2
assert ans2 == oracle_combinations(arr2, k2, x2)
print("Bài 2 OK")

[5, 4, 2, 1, 2, 3]
[4, 7, 8] 2
Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_solution(arr: List[int], k: int, x: int) -> List[int]:
    left, right = 0, len(arr) - k
    while left < right:
        mid = (left + right) >> 1
        if x - arr[mid] <= arr[mid + k] - x:
            right = mid
        else:
            left = mid + 1
    return arr[left:left + k]


assert exercise_3_solution([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert exercise_3_solution([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
assert exercise_3_solution([1, 3], 1, 2) == [1]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(arr: List[int], k: int, x: int) -> List[int]:
    l, r = 0, len(arr)
    while r - l > k:
        if x - arr[l] <= arr[r - 1] - x:
            r -= 1
        else:
            l += 1
    return arr[l:r]


assert exercise_4_solution([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert exercise_4_solution([0, 0, 1, 2, 3, 3, 4, 7, 8, 8], 3, 5) == [3, 3, 4]
assert exercise_4_solution([1, 2, 3], 3, 100) == [1, 2, 3]
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 10)
    arr = sorted(rng.randint(-6, 6) for _ in range(n))
    k = rng.randint(1, n)
    x = rng.randint(-8, 8)
    assert exercise_4_solution(arr, k, x) == closest_sort(arr, k, x)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_solution(arr: List[int], k: int, x: int) -> List[int]:
    left, right = 0, len(arr) - k
    while left < right:
        mid = (left + right) >> 1
        if x - arr[mid] < arr[mid + k] - x:      # hòa thì đi sang phải (số lớn hơn)
            right = mid
        else:
            left = mid + 1
    return arr[left:left + k]


def oracle_tie_larger(arr, k, x):
    return sorted(sorted(arr, key=lambda v: (abs(v - x), -v))[:k])


assert exercise_5_solution([1, 2, 3, 4, 5], 4, 3) == [2, 3, 4, 5]
assert exercise_5_solution([1, 3], 1, 2) == [3]
assert exercise_5_solution([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
rng = random.Random(5)
for _ in range(2000):
    n = rng.randint(1, 10)
    arr = sorted(rng.randint(-6, 6) for _ in range(n))
    k = rng.randint(1, n)
    x = rng.randint(-8, 8)
    assert exercise_5_solution(arr, k, x) == oracle_tie_larger(arr, k, x), (arr, k, x)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:
        left, right = 0, len(arr) - k             # điểm bắt đầu của cửa sổ, l thuộc [0, n - k]
        while left < right:
            mid = (left + right) >> 1
            if x - arr[mid] <= arr[mid + k] - x:  # đầu trái gần hơn hoặc hòa -> không dịch phải
                right = mid
            else:
                left = mid + 1
        return arr[left:left + k]
```

Câu thần chú: **"đáp án là cửa sổ dài k; so `x - arr[mid]` với `arr[mid + k] - x`; trái gần hơn hoặc hòa thì `right = mid`."**

### Checklist trước khi submit

- [ ] `right = len(arr) - k` (không phải `len(arr) - 1`), để `mid + k` luôn hợp lệ.
- [ ] Điều kiện dùng `<=` (hòa thì lấy số nhỏ hơn, tức bên trái).
- [ ] `while left < right` đi cùng `right = mid` và `left = mid + 1`.
- [ ] Trả về `arr[left:left + k]` (đã tăng dần, độ dài đúng `k`).
- [ ] Test: `k = 1`, `k = n`, `x` nhỏ hơn / lớn hơn cả mảng, phần tử trùng nhau, trường hợp hòa.
- [ ] Skeleton đúng: `class Solution`, method `findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]`.